# 01 — Exploratory Data Analysis

Load sample requests, embed them, and visualize with UMAP.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
%matplotlib inline

In [ ]:
df = pd.read_csv('../tests/fixtures/sample_requests.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

df['source'].value_counts().plot(kind='bar', ax=axes[0], title='Requests by Source')
df['mrr'].hist(bins=20, ax=axes[1])
axes[1].set_title('MRR Distribution')
df['text'].str.len().hist(bins=30, ax=axes[2])
axes[2].set_title('Request Text Length')

plt.tight_layout()
plt.show()

In [ ]:
from featrank.pipeline.cleaner import TextCleaner
from featrank.pipeline.embedder import Embedder
from featrank.schemas import RawRequest

requests = [RawRequest(**row) for row in df.to_dict('records')]

cleaner = TextCleaner()
kept, cleaned_texts = cleaner.clean_requests(requests)
print(f'After cleaning: {len(kept)} requests')

In [ ]:
embedder = Embedder(model_name='all-MiniLM-L6-v2')  # faster for notebook exploration
embeddings = embedder.embed(cleaned_texts)
print(f'Embedding shape: {embeddings.shape}')

In [ ]:
import umap

reducer = umap.UMAP(n_components=2, random_state=42, metric='cosine')
coords = reducer.fit_transform(embeddings)

plt.figure(figsize=(12, 8))
plt.scatter(coords[:, 0], coords[:, 1], alpha=0.5, s=20)
plt.title('UMAP projection of feature request embeddings (all-MiniLM-L6-v2)')
plt.xlabel('UMAP-1')
plt.ylabel('UMAP-2')
plt.tight_layout()
plt.show()

In [ ]:
from featrank.pipeline.clusterer import HDBSCANClusterer

clusterer = HDBSCANClusterer(min_cluster_size=5, min_samples=3)
clustered = clusterer.fit_predict(kept, embeddings, cleaned_texts)

labels = [r.cluster_id for r in clustered]
n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = labels.count(-1)
print(f'Clusters: {n_clusters}, Noise: {n_noise} ({n_noise/len(labels):.1%})')

In [ ]:
import matplotlib.cm as cm

unique_labels = sorted(set(labels))
colors = cm.tab20(np.linspace(0, 1, max(len(unique_labels), 1)))
color_map = {lbl: colors[i % len(colors)] for i, lbl in enumerate(unique_labels)}

plt.figure(figsize=(14, 9))
for lbl in unique_labels:
    mask = [l == lbl for l in labels]
    pts = coords[mask]
    color = 'lightgray' if lbl == -1 else color_map[lbl]
    label_str = 'noise' if lbl == -1 else f'cluster {lbl}'
    plt.scatter(pts[:, 0], pts[:, 1], c=[color], alpha=0.6, s=25, label=label_str)

plt.title('HDBSCAN clusters (UMAP projection)')
plt.xlabel('UMAP-1')
plt.ylabel('UMAP-2')
plt.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=7)
plt.tight_layout()
plt.show()